# UCITS Balanced Fund

This notebook presents a UCITS risk-monitoring workflow for a simulated balanced fund. The fund combines equity, fixed income, FX, cash, and derivative exposures within a UCITS-specific regulatory framework.

The analysis focuses on UCITS-relevant monitoring outputs, including global exposure, VaR monitoring, issuer and counterparty limits, eligibility checks, stress testing, and backtesting. UCITS monitoring is driven by UCITS regulatory rules and fund-specific policy settings, not by AIFMD or AIF-style risk frameworks.

> **Output gallery:** Tables and plots generated by this notebook are saved in the [`figs/ucits_balanced`](../../figs/ucits_balanced) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

> For supporting methodology notes, see [UCITS Balanced Notes](../../docs/notebooks_notes/ucits_balanced.md).

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# to initialize db if no tables exist abd to create session factory
from src.setup_db import run as setup_db
setup_db()
from src.data.database import get_engine
ENGINE = get_engine()

# initialize mock bloomberg API
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
BBG = Bloomberg()

# helper functions for risk computations and nice output formatting
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml

### Fund Example in this Notebook

The fund profile below sets the operating context for the risk workflow. It defines the strategy, fund type, base currency, reporting setup, and monitoring framework used by the calculations that follow.

<small>

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID = 'UCITS_Balanced'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated. They are used to show how a fund-level risk framework can be represented in a structured workflow.

---
### Risk management parameters 

The fund's monitoring parameters combine UCITS regulatory rules, here loaded as `ucits_limits` with fund-level configuration, here as `rmp`. 

**Regulatory Risk Parameters**

In [ ]:
# Load regulatory framework 
from src.data.reference_data import load_regulatory_framework

ucits_limits = load_regulatory_framework('ucits_regulatory_framework')

**Risk Management Policy**

Fund-level parameters define practical setup including benchmark selection, valuation date, reporting assumptions, and internal monitoring thresholds. The RMP is loaded as `rmp` and passed to the relevant risk functions. This keeps fund-specific parameters outside the calculation code. 

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="02",
)

In [ ]:
# read in RMP parameters for the fund
from src.data.reference_data import load_rmp
rmp = load_rmp(FUND_ID)

### Implementation Context

The analysis is performed as of a fixed valuation date, consistent with the point-in-time design used across the fund workflows.

In [ ]:
# fixed valuation date for all computations in this notebook
from src.config import VALUATION_DATE
VALUATION_DATE

Portfolio positions, fund characteristics, counterparty records, reference data, and market data are retrieved from the SQLite data layer. Market data is enriched through the simulated Bloomberg workflow before being passed to the risk analytics modules.

For a fuller explanation of the data workflow, see the [Data Layer Workflow](../data_workflows/01_data_layer_workflow.ipynb).

In [ ]:
# enrich fund postion data w/ info from Bloomberg
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# display the enriched risk dataframe first 2 rows
risk_df.head(2)

From this point onward, the notebook uses helper functions to render summary tables and HTML views. Data aggregation, calculations, and reporting logic remain inside the package modules, so the notebook stays focused on review and interpretation.

## 1. Load Positions

Load today's UCITS Balanced Fund positions from the daily fund
administrator export.

In [ ]:
# query from SQLdb
from src.data.database import query_positions
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)

# enrich w/ info from Bloomberg
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# info in risk_df covers many risk inputs
NAV = risk_df['market_value_eur'].sum()

phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV)

In [ ]:
phtml.display_top_positions(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")

In [ ]:
phtml.display_asset_class_breakdown(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")  # Also computes NAV internally

# 2. Validate Positions
UCITS imposes strict eligibility and concentration rules that must be checked at the position level before any risk calculation begins.

In [ ]:
# UCITS position-level compliance checks
from src.risk.ucits_compliance_checks import run_ucits_compliance_checks

compliance_result = run_ucits_compliance_checks(positions, NAV)
phtml.display_ucits_compliance_checks(compliance_result, export_id="02a", fund_id=FUND_ID, valuation_date=VALUATION_DATE)

## 2. VaR and Expected Shortfall

Market risk is monitored through Historical VaR and Expected Shortfall, as per registered in the RMP. The confidence level, holding period, and distribution assumptions are also taken from the fund's RMP and applied to the current portfolio snapshot. 

In [ ]:
# get regulatory parameters related to var
ucits_var = ucits_limits['var_framework']

var_confidence = ucits_var['confidence_level']
var_holding_period = ucits_var['holding_period_days']
var_lookback = ucits_var['lookback_window_days']
absolute_var_limit_pct = ucits_var['absolute_limit_pct']
relative_var_limit = ucits_var['relative_limit_multiplier']
relative_var_reference = ucits_var['relative_limit_reference_portfolio']

# Unpack risk parameters related to var from RMP
rmp_var = rmp['var_framework']
df = rmp_var['parametric_degrees_of_freedom']

In [ ]:
# 1-day fixed-position VaR + ES computation (1 day and scaled)
from src.pipeline.fixed_position_var import compute_fixed_position_var_1day

var_result = compute_fixed_position_var_1day(
    engine=ENGINE,
    fund_id=FUND_ID,
    valuation_date=VALUATION_DATE,
    confidence=var_confidence,
    df=df,
    horizon=var_holding_period,
)

# Display VaR and ES (extracts nav, horizon, and valuation_date from var_result)
phtml.display_var_es(var_result, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="06")

---

## 4. Relative VaR

The UCITS relative VaR limit (where applicable) requires that the fund VaR does not exceed 2× the VaR of the reference portfolio.

$$	ext{Relative VaR} = \frac{VaR_{fund}}{VaR_{reference}} \leq 2$$

The reference portfolio for this UCITS Balanced fund is a 60/40 allocation (60% MSCI World, 40% EUR government bonds), documented in the fund's risk management policy.

In [ ]:
from src.risk.ucits_relative_var import compute_ucits_relative_var_point_in_time
rel_var_result = compute_ucits_relative_var_point_in_time(
    ENGINE, FUND_ID, var_result, var_confidence, var_lookback, var_holding_period, relative_var_limit, BBG, VALUATION_DATE, rmp
)

phtml.display_ucits_relative_var_point_in_time(
    rel_var_result,
    valuation_date=VALUATION_DATE,
    fund_id=FUND_ID,
    export_id="09"
)


## 5. SRRI Computation
The SRRI (Synthetic Risk and Reward Indicator) is computed from 5 years of weekly NAV returns,
annualised, and mapped to a 1–7 category per CESR/10-673.

CESR/10-673 is the guidelines document published by the Committee of European Securities Regulators (the predecessor to ESMA) that defines exactly how to compute the SRRI. It specifies 
* the 5-year weekly return window
* the annualisation formula
* seven volatility buckets that map to categories 1 through 7. 

It is the regulatory basis for the risk indicator shown on every UCITS KIID.

| SRRI | Volatility range |
|------|-----------------|
| 1    | < 0.5%          |
| 2    | 0.5% – 2%       |
| 3    | 2% – 5%         |
| 4    | 5% – 10%        |
| 5    | 10% – 15%       |
| 6    | 15% – 25%       |
| 7    | >= 25%          |

In [ ]:
from src.risk.ucits_srri import compute_srri_from_fund

srri_result = compute_srri_from_fund(ENGINE, FUND_ID)

phtml.display_ucits_srri_point_in_time(
    srri_result,
    valuation_date=VALUATION_DATE,
    fund_id=FUND_ID,
    export_id="10"
)

srri = srri_result['sri_bucket']


## 6. UCITS Stress Testing

UCITS regulations (2009/65/EC and ESMA guidelines) require stress testing but do not
prescribe specific scenario parameters. Shocks below are defined in the fund's Risk
Management Policy (RMP) and approved by the board.

Scenarios covered:
- **Equity crash**: equity down 30%
- **Rate shock**: parallel shift up 200bps
- **Credit widening**: credit spreads widen 150bps
- **FX stress**: USD/GBP depreciate 15% vs EUR
- **Combined**: simultaneous equity, rate, credit and FX shock
- **Historical**: 2008 financial crisis, 2020 COVID crash, 2022 rate shock

In [ ]:
from src.risk.risk_utils import HISTORICAL_SCENARIOS
phtml.display_historical_scenarios(HISTORICAL_SCENARIOS, fund_id=FUND_ID, export_id="11")

In [ ]:
custom_scenarios = {
    'Equity Crash -30%'      : rk.stress_equity(risk_df, delta_equity=-0.30),
    'Rate Shock +200bps'     : rk.stress_rates(risk_df, delta_y=0.02),
    'Credit Widening +150bps': rk.stress_credit(risk_df, delta_spread=0.015),
    'FX Stress USD/GBP -15%' : rk.stress_fx(risk_df, fx_shocks={'USD': -0.15, 'GBP': -0.15}),
    'Combined'               : rk.stress_combined(risk_df),
}
    
phtml.display_scenarios(risk_df, custom_scenarios, add_historical=True, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="12")

In [ ]:
# MRS-42: stress testing setup
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)
NAV     = risk_df['market_value_eur'].sum()


In [ ]:
# Stress scenarios from fund RMP
custom_scenarios = {
    'Equity Crash -30%'      : rk.stress_equity(risk_df, delta_equity=-0.30),
    'Rate Shock +200bps'     : rk.stress_rates(risk_df, delta_y=0.02),
    'Credit Widening +150bps': rk.stress_credit(risk_df, delta_spread=0.015),
    'FX Stress USD/GBP -15%' : rk.stress_fx(risk_df, fx_shocks={'USD': -0.15, 'GBP': -0.15}),
    'Combined'               : rk.stress_combined(risk_df),
}

# Display with historical scenarios
phtml.display_scenarios(risk_df, custom_scenarios, add_historical=True, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="06")

In [ ]:
# Combined scenario
cb = stress_combined(risk_df)
cb_pct = cb['stressed_pnl_eur'] / NAV * 100

components = {
    'Equity': cb['equity_pnl'] / NAV * 100,
    'Rates':  cb['rates_pnl']  / NAV * 100,
    'Credit': cb['credit_pnl'] / NAV * 100,
    'FX':     cb['fx_pnl']     / NAV * 100,
}

# Historical scenarios
scenarios = ['2008', '2020', '2022']
labels    = ['2008 Financial Crisis', '2020 COVID Crash', '2022 Rate Shock']
results   = [stress_historical(risk_df, s) for s in scenarios]
pnls_pct  = [r['stressed_pnl_eur'] / NAV * 100 for r in results]

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                                gridspec_kw={'width_ratios': [4, 3]})

# ── left: combined scenario ──────────────────────────────────────────────────
bars1 = ax1.bar(components.keys(), components.values(),
                color=[ACCENT2 if v < 0 else ACCENT3 for v in components.values()],
                width=0.45, alpha=0.85)
ax1.axhline(0, color=C['dim'], lw=0.8)
ax1.set_ylabel('P&L impact (% NAV)', fontsize=11)
ax1.tick_params(axis='both', labelsize=11)

# right
section_title(ax1, f'Combined Stress Total: {cb_pct:.2f}% NAV', fontsize=12)
for bar, val in zip(bars1, components.values()):
    y = val - 0.3 if abs(val) < 2 else val / 2
    va = 'top' if abs(val) < 2 else 'center'
    ax1.text(bar.get_x() + bar.get_width() / 2, y,
             f'{val:.2f}%', ha='center', va=va,
             fontsize=11, color='white', fontweight='bold')

# ── right: historical scenarios ──────────────────────────────────────────────
bars2 = ax2.bar(labels, pnls_pct,
                color=[ACCENT2 if v < 0 else ACCENT3 for v in pnls_pct],
                width=0.45, alpha=0.85)
ax2.axhline(0, color=C['dim'], lw=0.8)
ax2.set_ylabel('P&L impact (% NAV)', fontsize=11)
ax2.tick_params(axis='both', labelsize=11)
section_title(ax2, 'Historical Stress Scenarios', fontsize=12)
for bar, val in zip(bars2, pnls_pct):
    ax2.text(bar.get_x() + bar.get_width() / 2, val / 2,
             f'{val:.2f}%', ha='center', va='center',
             fontsize=11, color='white', fontweight='bold')

sup_title(fig, f'Stress Tests — {FUND_ID}  |  {VALUATION_DATE}', fontsize=18)
plt.tight_layout()
save_fig(fig, FUND_ID, '03. Stress tests: shock, combined, historical')
plt.show()

print(f"Combined stress P&L: EUR {cb['stressed_pnl_eur']:,.0f}  ({cb_pct:.2f}% NAV)")
for label, r, pct in zip(labels, results, pnls_pct):
    print(f"{label:30s}: EUR {r['stressed_pnl_eur']:>15,.0f}  ({pct:.2f}% NAV)")

> **Methodology note**: stress P&L is computed using first-order sensitivities
> (delta for equities, modified duration for rates and credit, direct revaluation for FX).
> Convexity and cross-gamma effects are not captured. In production, a ManCo would
> source these figures from a risk system (Bloomberg PORT, Aladdin, Axioma) which
> performs full revaluation. Results here should be read as directional estimates.

---

## 7. VaR Backtest

VaR backtesting compares predicted VaR against realised daily P&L over a 250-day rolling window. Two statistical tests are applied:

- **Kupiec POF**: tests whether the breach rate equals the expected rate (1% for 99% VaR)
- **Christoffersen**: tests whether breaches are independently distributed

Zone classification (Basel traffic light):
- **Green** (0-4 breaches): model acceptable
- **Amber** (5-9 breaches): explanation required
- **Red** (10+ breaches): model must be revised

Backtesting supports internal VaR model validation and monitoring.

In [ ]:
# VaR Backtest — rolling 1-day VaR over lookback window
from src.risk.var_backtest import compute_var_backtest_rolling, create_backtest_report
import pandas as pd

# Compute rolling VaR backtest
start_date = (pd.Timestamp(VALUATION_DATE) - pd.tseries.offsets.BDay(var_lookback)).strftime('%Y-%m-%d')

backtest_df = compute_var_backtest_rolling(
    engine=ENGINE,
    fund_id=FUND_ID,
    start_date=start_date,
    end_date=VALUATION_DATE,
    lookback=var_lookback,
)

print(f"✓ Backtest computed: {len(backtest_df)} trading days")

# Create report with Kupiec & Christoffersen tests
report = create_backtest_report(backtest_df)

# Display using HTML table
phtml.display_backtest_report(report, window_size=var_lookback, valuation_date=VALUATION_DATE, model="Historical", fund_id=FUND_ID, export_id="07")


In [ ]:
# VaR Backtest Plot — P&L vs VaR with breaches highlighted
from src.ui.backtest_plot import plot_var_backtest

# Extract aligned returns and VaR
returns_shifted = backtest_df['realised_return'].iloc[1:].values
var_shifted = backtest_df['var_99_pct'].iloc[:-1].values
dates_shifted = backtest_df['date'].iloc[1:].values

# Extract test p-values
kupiec_pvalue = report[report['confidence'] == var_confidence*100]['kupiec_p'].values[0] if len(report[report['confidence'] == var_confidence*100]) > 0 else None
chris_pvalue = report[report['confidence'] == var_confidence*100]['christoffersen_p'].values[0] if len(report[report['confidence'] == var_confidence*100]) > 0 else None

fig, ax = plot_var_backtest(
    dates_shifted,
    returns_shifted,
    var_shifted,
    FUND_ID,
    title=f'VaR Backtest — {FUND_ID} ({var_confidence*100:.0f}% confidence, {var_lookback}-day lookback)',
    kupiec_pvalue=kupiec_pvalue,
    christoffersen_pvalue=chris_pvalue,
    valuation_date=VALUATION_DATE,
    export_id="08"
)


## 8. Absolute and Relative VaR Monitoring Report

In [ ]:
# MRS-44: VaR monitoring report
from src.risk.risk_utils import var_scale

# --- Absolute VaR ---
pnl_1y        = returns[-250:]
abs_var_1d    = var_historical(pnl_1y, confidence=0.99)
abs_var_20d   = var_scale(abs_var_1d, horizon=20) * 100  # % NAV
abs_limit     = 20.0
abs_util      = abs_var_20d / abs_limit * 100

# --- Relative VaR ---
rel_var_current = rolling_ratio.iloc[-1] if len(rolling_ratio) > 0 else None
rel_limit       = 2.0
rel_util        = rel_var_current / rel_limit * 100

# --- Summary table ---
summary = pd.DataFrame([
    {'Metric': 'Absolute VaR (20d 99%)',    'Value': f'{abs_var_20d:.2f}%',       'Limit': '20.00%',  'Utilisation': f'{abs_util:.1f}%',  'Status': 'OK' if abs_var_20d < abs_limit else 'BREACH'},
    {'Metric': 'Relative VaR (ratio)',       'Value': f'{rel_var_current:.2f}x',   'Limit': '2.00x',   'Utilisation': f'{rel_util:.1f}%',  'Status': 'OK' if rel_var_current < rel_limit else 'BREACH'},
    {'Metric': 'SRRI',                       'Value': str(srri),                   'Limit': '—',       'Utilisation': '—',                 'Status': '—'},
    {'Metric': 'ESMA Zone (250d)',            'Value': zone_250,                    'Limit': 'Green',   'Utilisation': f'{n_250} breaches', 'Status': 'OK' if zone_250 == 'Green' else 'REVIEW'},
])
summary.set_index('Metric', inplace=True)
summary


In [ ]:
var_lim = 100
var_warning = 80

metrics  = ['Absolute VaR', 'Relative VaR']
utils    = [abs_util, rel_util]
colors   = [ACCENT2 if u > 80 else ACCENT3 if u > 60 else ACCENT for u in utils]

fig, ax = plt.subplots(figsize=(8, 2))
bars = ax.barh(metrics, utils, color=colors, height=0.4, alpha=0.85)

ax.axvline(var_lim, color=ACCENT2, lw=1.5, linestyle='--')
ax.text(var_lim - 1, 0.5, 'Hard limit', color=ACCENT2, fontsize=8,
        va='center', ha='right', rotation=90,
        transform=ax.get_xaxis_transform())

ax.axvline(var_warning, color=C['amber'], lw=1, linestyle='--')
ax.text(var_warning - 1, 0.5, 'Warning', color=C['amber'], fontsize=8,
        va='center', ha='right', rotation=90,
        transform=ax.get_xaxis_transform())

ax.set_xlabel('Limit utilisation (%)', fontsize=9)
section_title(ax, f'VaR Limit Utilisation — {VALUATION_DATE}')
ax.grid(False)
for bar, val in zip(bars, utils):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
save_fig(fig, FUND_ID, "06. VAR Limit Utilisation  - UCITS")
plt.show()



## 9. SRRI Monitoring and KIID Update Trigger

CESR/10-673 requires ManCos to monitor SRRI continuously and update the KIID
if the SRRI changes category for 4 consecutive months. The rolling SRRI is
computed monthly using a trailing 260-week (5-year) window.

In [ ]:
# MRS-45: SRRI monitoring
nav_history_full = query_nav_history(ENGINE, FUND_ID)
nav_history_full['date'] = pd.to_datetime(nav_history_full['date'])
nav_history_full = nav_history_full.set_index('date')

# resample to weekly, compute rolling 260-week SRRI monthly
weekly_nav_full = nav_history_full['nav_eur'].resample('W').last()
weekly_ret_full = weekly_nav_full.pct_change().dropna()

# compute SRRI at each month-end using trailing 260 weeks
monthly_ends = weekly_ret_full.resample('ME').last().index
rolling_srri = []

for dt in monthly_ends:
    window_ret = weekly_ret_full[:dt].iloc[-260:]
    if len(window_ret) < 52:
        continue
    sigma_w   = window_ret.std()
    sigma_a   = sigma_w * np.sqrt(52)
    rolling_srri.append({'date': dt, 'srri': compute_srri(sigma_a), 'sigma_ann': sigma_a})

srri_df = pd.DataFrame(rolling_srri).set_index('date')
print(f"Rolling SRRI observations : {len(srri_df)}")
srri_df.tail(6)

In [ ]:
# flag 4 consecutive months where SRRI stays at a new level
srri_df['srri_prev']     = srri_df['srri'].shift(1)
srri_df['changed']       = srri_df['srri'] != srri_df['srri_prev']

# count consecutive months at current SRRI vs initial SRRI
initial_srri             = srri_df['srri'].iloc[0]
srri_df['consec_new']    = 0

consec = 0
ref    = srri_df['srri'].iloc[0]
for idx, row in srri_df.iterrows():
    if row['srri'] != ref:
        consec += 1
    else:
        consec = 0
        ref    = row['srri']
    srri_df.at[idx, 'consec_new'] = consec

srri_df['kiid_update'] = srri_df['consec_new'] >= 4

n_triggers = srri_df['kiid_update'].sum()
print(f"KIID update triggers in history : {n_triggers}")
print(f"Current SRRI                    : {srri_df['srri'].iloc[-1]}")
print(f"KIID update required now        : {srri_df['kiid_update'].iloc[-1]}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
sup_title(fig, 'SRRI Monitoring', fontsize=15)

# top: rolling annualised volatility
ax1.plot(srri_df.index, srri_df['sigma_ann'] * 100,
      color=ACCENT, lw=1.2, label='Annualised vol (%)')
ax1.set_ylabel('Volatility (%)', fontsize=9)
ax1.grid(True, axis='y', alpha=0.15, linestyle='--')
ax1.tick_params(labelsize=9, length=0)
ax1.legend(fontsize=8)

# bottom: rolling SRRI with KIID triggers
ax2.step(srri_df.index, srri_df['srri'],
             color=ACCENT, lw=1.5, label='SRRI', where='post')
ax2.set_yticks([1, 2, 3, 4, 5, 6, 7])
ax2.set_ylim(0.5, 7.5)
ax2.set_ylabel('SRRI category', fontsize=9)
ax2.grid(True, axis='y', alpha=0.15, linestyle='--')
ax2.tick_params(labelsize=9, length=0)

triggers = srri_df[srri_df['kiid_update']]
if len(triggers) > 0:
    ax2.scatter(triggers.index, triggers['srri'],
                    color=ACCENT2, s=5, zorder=5,
                    label=f'KIID update trigger ({len(triggers)})')
ax2.legend(fontsize=8)

plt.tight_layout()
save_fig(fig, FUND_ID, "07. SRRI Monitoring")
plt.show()



## 10. Monthly Risk Report

Summary risk report as of the most recent valuation date.
Produced monthly by the Risk Management function and reviewed by the Board.

In [ ]:
# MRS-46: monthly risk report
from datetime import datetime

report_date = pd.Timestamp(VALUATION_DATE).strftime('%B %d, %Y')
nav_eur     = risk_df['market_value_eur'].sum()

print('=' * 60)
print(f'  UCITS BALANCED FUND — MONTHLY RISK REPORT')
print(f'  Valuation date : {report_date}')
print(f'  NAV            : EUR {nav_eur:,.0f}')
print('=' * 60)

print('\n  1. VAR SUMMARY')
print('  ' + '-' * 40)
print(f'  Absolute VaR (20d 99%)  : {abs_var_20d:.2f}%   limit 20.00%   util {abs_util:.1f}%')
print(f'  Relative VaR (ratio)    : {rel_var_current:.2f}x     limit 2.00x     util {rel_util:.1f}%')
print(f'  VaR model               : Historical simulation, 250-day window')

print('\n  2. SRRI')
print('  ' + '-' * 40)
print(f'  Current SRRI            : {srri_df["srri"].iloc[-1]}')
print(f'  Annualised volatility   : {srri_df["sigma_ann"].iloc[-1]*100:.2f}%')
print(f'  KIID update required    : {"YES — action required" if srri_df["kiid_update"].iloc[-1] else "No"}')

print('\n  3. BACKTEST (ESMA)')
print('  ' + '-' * 40)
print(f'  Observation window      : 250 trading days')
print(f'  Breaches                : {n_250}')
print(f'  Expected breaches       : 2.5 (1% x 250)')
print(f'  ESMA zone               : {zone_250}')
print(f'  Kupiec p-value          : {report["kupiec_p"].iloc[0]:.4f}')
print(f'  Christoffersen p-value  : {report["christoffersen_p"].iloc[0]:.4f}')

print('\n  4. STRESS TESTING')
print('  ' + '-' * 40)
print(f'  Equity crash -30%       : {eq_pct:.2f}% NAV')
print(f'  Rate shock +200bps      : {rt_pct:.2f}% NAV')
print(f'  Credit widening +150bps : {cr_pct:.2f}% NAV')
print(f'  FX stress -15%          : {fx_pct:.2f}% NAV')
print(f'  Combined scenario       : {cb_pct:.2f}% NAV')
print(f'  2008 financial crisis   : {pnls_pct[0]:.2f}% NAV')
print(f'  2020 COVID crash        : {pnls_pct[1]:.2f}% NAV')
print(f'  2022 rate shock         : {pnls_pct[2]:.2f}% NAV')

print('\n  5. COMPLIANCE')
print('  ' + '-' * 40)
abs_status = 'COMPLIANT' if abs_var_20d < abs_limit else 'BREACH'
rel_status = 'COMPLIANT' if rel_var_current < rel_limit else 'BREACH'
print(f'  Absolute VaR limit      : {abs_status}')
print(f'  Relative VaR limit      : {rel_status}')
print(f'  ESMA backtest zone      : {zone_250}')
print(f'  UCITS eligible          : Yes')

print('\n' + '=' * 60)
print('  Prepared by  : Risk Management')
print(f'  Report date  : {pd.Timestamp(VALUATION_DATE).strftime("%B %d, %Y")}')
print('=' * 60)

<!-- Blank markdown cell retained to preserve notebook structure during markdown-only cleanup. -->

In [ ]:
# MRS-31: Redemption stress — UCITS Balanced
# Compute liquidity buckets (risk_df already defined in Section 6)
_ucits_liq = days_to_liquidate(risk_df, pct_adv=0.25)
_ucits_liq = liquidity_buckets(_ucits_liq)
NOTICE = 5   # contractual notice period (days)
_SCENARIOS = [
    (0.10, 'Normal    (10% NAV)'),
    (0.25, 'Large     (25% NAV)'),
    (0.50, 'Stress    (50% NAV)'),
]

print(f'Fund: UCITS Balanced  |  NAV: EUR {NAV:,.0f}  |  Notice: {NOTICE} days')
print()
print(f"{'':22} {'Redemption (M)':>14} {'Liquid (M)':>12} {'Gap (M)':>12} {'Coverage':>10} Action")
print('\u2500' * 95)

_redstress = {}
for _pct, _label in _SCENARIOS:
    _r = redemption_stress(_ucits_liq, NAV, redemption_pct=_pct, notice_days=NOTICE)
    _redstress[_pct] = _r
    _gap = f"+{_r['liquidity_gap_eur']/1e6:.1f}M" if _r['liquidity_gap_eur'] >= 0 else f"{_r['liquidity_gap_eur']/1e6:.1f}M"
    print(f"{_label:<22} {_r['redemption_amount_eur']/1e6:>13.1f}M {_r['liquid_assets_eur']/1e6:>11.1f}M "
          f"{_gap:>12} {_r['coverage_ratio']:>9.2f}x  {_r['recommendation']}")
print('\u2500' * 95)
print('Largest-investor stress: see Section 6.3')

### 6.3 Investor Concentration

Investor concentration is monitored as part of the UCITS operational and liquidity risk workflow.

For a daily-dealing UCITS, a concentrated investor base can affect liquidity management because large subscriptions or redemptions may require portfolio trading, cash-buffer usage, or escalation to the management company.

The investor register is simulated. The output should be interpreted as an investor-base monitoring indicator, not as an AIF-style Annex IV concentration measure.

In [ ]:
# MRS-32: Investor concentration — UCITS Balanced
# UCITS: 500+ retail investors — representative sample, top 10 + aggregate
_investors = pd.DataFrame([
    {'investor_id': 'UC001', 'investor_name': 'Pension Plan A',           'investor_type': 'Pension Plan', 'aum_eur': NAV * 0.0200},
    {'investor_id': 'UC002', 'investor_name': 'Distribution Platform B',  'investor_type': 'Platform',     'aum_eur': NAV * 0.0170},
    {'investor_id': 'UC003', 'investor_name': 'Insurance Wrapper C',      'investor_type': 'Insurance',    'aum_eur': NAV * 0.0130},
    {'investor_id': 'UC004', 'investor_name': 'Retail Investor D',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0080},
    {'investor_id': 'UC005', 'investor_name': 'Retail Investor E',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0070},
    {'investor_id': 'UC006', 'investor_name': 'Retail Investor F',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0065},
    {'investor_id': 'UC007', 'investor_name': 'Retail Investor G',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0060},
    {'investor_id': 'UC008', 'investor_name': 'Retail Investor H',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0055},
    {'investor_id': 'UC009', 'investor_name': 'Retail Investor I',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0050},
    {'investor_id': 'UC010', 'investor_name': 'Retail Investor J',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0045},
    {'investor_id': 'UC_REM','investor_name': 'Remaining (490+ retail)', 'investor_type': 'Retail',       'aum_eur': NAV * 0.9075},
])

_conc = investor_concentration(_investors, NAV, threshold=0.20)
_top  = _conc['by_investor']
_type = _investors.set_index('investor_id')['investor_type']

print(f'Investor Concentration — UCITS Balanced  |  NAV: EUR {NAV:,.0f}')
print('ESMA threshold: 20% single / 50% top-3\n')
print(f"{'':4} {'Investor':<30} {'Type':<18} {'AUM (EUR)':>14} {'% NAV':>8}")
print('\u2500' * 80)
for _rank, (_, _row) in enumerate(_top.iterrows(), 1):
    _t = _type.get(_row['investor_id'], '')
    print(f"{_rank:<4} {_row['investor_name']:<30} {_t:<18} {_row['aum_eur']:>14,.0f} {_row['pct_nav']*100:>7.1f}%")
print('\u2500' * 80)

_flag_s = '\u26a0 ESMA flag'       if _conc['concentration_flag'] else '\u2713 OK'
_flag_3 = '\u26a0 High conc.'      if _conc['high_concentration'] else '\u2713 OK'
print(f"\nLargest investor : {_conc['largest_investor_pct']*100:.1f}% NAV  {_flag_s}")
print(f"Top 3 investors  : {_conc['top3_pct']*100:.1f}% NAV  {_flag_3}")

# Largest-investor redemption stress (4th scenario)
_r4   = redemption_stress(_ucits_liq, NAV, redemption_pct=_conc['largest_investor_pct'], notice_days=5)
_gap4 = f"+{_r4['liquidity_gap_eur']/1e6:.1f}M" if _r4['liquidity_gap_eur'] >= 0 else f"{_r4['liquidity_gap_eur']/1e6:.1f}M"
print(f"\nLargest-investor stress ({_conc['largest_investor_pct']*100:.1f}% NAV, 5-day notice):")
print(f"  Redemption : EUR {_r4['redemption_amount_eur']:,.0f}")
print(f"  Liquid     : EUR {_r4['liquid_assets_eur']:,.0f}")
print(f"  Gap        : {_gap4}  |  Coverage: {_r4['coverage_ratio']:.2f}x")
print(f"  Action     : {_r4['recommendation']}")

print('\nMonitoring recommendation:')
if _conc['high_concentration']:
    print('  \u2014 Enhanced monitoring: top-3 investors represent significant co-ordinated exit risk')
    print('  \u2014 Maintain liquidity buffer >= largest investor AUM')
if _conc['concentration_flag']:
    print(f'  \u2014 Gate-trigger review: largest investor at {_conc["largest_investor_pct"]*100:.1f}% NAV')
if not _conc['concentration_flag'] and not _conc['high_concentration']:
    print('  \u2014 No immediate action. Continue quarterly investor concentration monitoring.')

---

### 6.4 Counterparty Risk

UCITS Article 52 caps OTC derivative counterparty exposure at 10% of NAV for credit institutions and 5% for all others. The relevant exposure is the net uncollateralised position after netting initial and variation margin held at the counterparty.

> Collateral coverage is simulated. A production system would derive these figures from the daily collateral reconciliation.

In [ ]:
# MRS-XX: Counterparty risk — UCITS
# Simulated OTC derivatives counterparty register
_cp_ucits = rk.load_counterparty(FUND_ID)
_cp_ucits['exposure_eur']     = _cp_ucits['exposure_pct'] * NAV
_cp_ucits['collateral_eur']   = _cp_ucits['exposure_eur'] * _cp_ucits['collateral_cover']
_cp_ucits['net_exposure_eur'] = _cp_ucits['exposure_eur'] * (1 - _cp_ucits['collateral_cover'])
_cp_ucits['net_pct_nav']      = _cp_ucits['net_exposure_eur'] / NAV
_cp_ucits['limit_pct']        = _cp_ucits['type'].map(
                                    {'credit_institution': 0.10, 'other': 0.05})
_cp_ucits['breach']           = _cp_ucits['net_pct_nav'] > _cp_ucits['limit_pct']

_worst_cp    = _cp_ucits.loc[_cp_ucits['net_exposure_eur'].idxmax()]
_cp_loss_eur = _worst_cp['net_exposure_eur']
_cp_loss_pct = _worst_cp['net_pct_nav']

phtml.display_counterparty_risk_ucits(NAV, _cp_ucits, _worst_cp, _cp_loss_eur, _cp_loss_pct)

---

## 12. Pre-Trade Compliance Check

For UCITS, pre-trade compliance is a statutory obligation. UCITSD Articles 50, 52, and 83 impose hard limits that a ManCo cannot waive.

`pre_trade_check` runs the following checks:
- **Absolute VaR** post-trade < 20% NAV (20-day, 99%)
- **Relative VaR** < 2× reference portfolio
- **5/10/40 rule** (Article 52): single issuer < 10% NAV; issuers > 5% aggregate to < 40%
- **Eligible assets** (Article 50): no direct real estate, loans, CLOs, private equity
- **Counterparty exposure** (OTC derivatives): < 10% for credit institutions, < 5% otherwise
- **Borrowing limit** < 10% NAV (Article 83)

> **ETF look-through limitation.** This portfolio is dominated by broad index ETFs (SPY 48%, Euro Stoxx 28%). In a fully compliant implementation, these ETFs would be evaluated on a look-through basis. The simplified `pre_trade_check` applies the 5/10/40 rule at ETF level. The relevant compliance question for each scenario below is whether the *proposed trade* introduces a new or worsened breach relative to the current portfolio state.

In [ ]:
from src.risk.risk_utils import pre_trade_check

### 12.1  Scenario 1 — Small IG bond buy

Buy BMW AG bonds, new issuer not currently in the portfolio. \
EUR 5M notional on a EUR 515M NAV = 0.97% of NAV. \
The trade is small and compliant. The check will surface the pre-existing SPY and \
Euro Stoxx concentrations (ETF look-through limitation), but the proposed trade does not \
introduce any new or worsened breach — which is the material compliance question.\

In [ ]:
# Pre-trade check context (UCITS version — no derivatives)

PTC_CTX = dict(
    engine            = ENGINE,
    fund_id           = FUND_ID,
    date              = VALUATION_DATE,
    counterparties_df = _cp_ucits,
    bbg               = BBG,
)

In [ ]:
# BMW AG IG bond — new issuer, small position, eligible instrument
trade_bond = {
    'isin'           : 'XS2600000001',
    'direction'      : 'buy',
    'quantity'       : 5_000_000,         # EUR 5 M at par
    'price_eur'      : 1.00,
    'asset_class'    : 'Bond',
    'sub_asset_class': 'IG Corporate',
    'dur_adj_mid'    : 3.5,
    'currency'       : 'EUR',
    'adv_eur'        : 10_000_000,
    'issuer'         : 'BMW AG',}



result_conc = pre_trade_check(trade_bond, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_conc)

### 12.2  Scenario 2 — Single-stock concentration breach (5/10/40 rule, Article 52)

Buy Apple Inc (not in the UCITS portfolio), 350,000 shares at EUR 172/share \
= EUR 60.2M notional. On NAV of EUR 514.8M, this is **11.7%** of NAV — \
a clear breach of the 10% single-issuer hard limit under UCITSD Article 52. \
A UCITS ManCo must block this order.

The 10% limit applies per *issuer* of transferable securities, not per instrument. \
If the same issuer were held in both equity and bond form, both exposures would be \
aggregated for the 5/10/40 test. A UCITS portfolio can hold up to six issuers at 5–10% \
(the "40 rule" caps the aggregate of all positions > 5%), but no single issuer may exceed 10%.

In [ ]:
# Apple Inc — new single-stock position sized to breach 10% issuer limit (11.7% NAV)
trade_conc = {
    'isin'           : 'US0231351067',
    'direction'      : 'buy',
    'quantity'       : 350_000,           # 350 k × EUR 172 = EUR 60.2 M
    'price_eur'      : 172.00,
    'asset_class'    : 'Equity',
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.25,
    'currency'       : 'USD',
    'adv_eur'        : 200_000_000,
    'issuer'         : 'Apple Inc',
}

result_conc = pre_trade_check(trade_conc, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_conc)

### 12.3  Scenario 3 — Ineligible asset (UCITSD Article 50)

The portfolio manager proposes to invest EUR 10M in a direct real estate holding. \
Direct property is not a transferable security, money market instrument, unit in a UCITS/UCI, \
bank deposit, or eligible derivative — it is therefore ineligible under UCITSD Article 50. \
A UCITS ManCo must refuse this trade. There is no discretion here; this is a statutory \
prohibition, not an internal risk limit.

In [ ]:
# Direct property — ineligible under UCITSD Art. 50
trade_inelig = {
    'isin'           : 'LU-PROP-DIRECT-001',
    'direction'      : 'buy',
    'quantity'       : 1,
    'price_eur'      : 10_000_000,
    'asset_class'    : 'Real Estate',
    'sub_asset_class': 'Direct Property',
    'currency'       : 'EUR',
    'adv_eur'        : 0,
    'issuer'         : 'Luxembourg Office Holdings SA',
}

result_inelig = pre_trade_check(trade_inelig, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_inelig)

---

## 13. ESG Risk Indicators

The notebook computes portfolio-level ESG indicators using NAV-weighted averages and the fund's internal ESG threshold.

Metrics monitored include weighted average ESG score, percentage of NAV below the internal threshold, controversy flags, and carbon intensity.

ESG scores for liquid instruments are sourced from Bloomberg. Instruments without a Bloomberg ticker use fund administrator ESG data.

ESG scores should be treated as sustainability-risk or disclosure indicators unless explicitly mapped to SFDR Article 6, 8, or 9 concepts.

In [ ]:
# esg_df  = build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
# summary = esg_portfolio_summary(esg_df, NAV)

import src.risk.esg_utils as esg_u 
esg_df  = esg_u.build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
esg_u.display_esg_assets(esg_df)

In [ ]:
esg_u.display_esg_summary(esg_df)

In [ ]:
esg_u.plot_esg_profile(esg_df, FUND_ID, plot_title='ESG profile - UCITS Balanced')

## 11. P&L Attribution by Risk Factor

P&L attribution explains daily portfolio return drivers by risk factor. In this notebook, it is used as an internal model review and governance output for the UCITS risk workflow.

The value bridge methodology decomposes daily P&L into:

- market beta
- rates
- FX
- residual or unexplained P&L

Benchmark: SX5E (Euro Stoxx 50). The benchmark is used as the equity-market factor in the attribution model for this simulated EUR-denominated balanced fund.

The residual captures P&L not explained by the selected market, rates, and FX factors.

> Sensitivity-based attribution uses daily positions and daily market moves, consistent with the position-based risk workflow used in the notebook.

In [ ]:
import sqlalchemy as sa
from src.risk.risk_utils import compute_pnl_attribution
from src.data.database import query_nav_history

# Actual daily P&L
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])
nav_history = nav_history.set_index('date').sort_index()
pnl_actual  = nav_history['pnl_eur'].dropna()

START_DATE         = pnl_actual.index.min().strftime('%Y-%m-%d')
VALUATION_DATE_STR = VALUATION_DATE

# Benchmark — SPY for USD-heavy long/short portfolio
spy_bm = BBG.bdh('SPY US Equity', 'PX_LAST', START_DATE, VALUATION_DATE_STR)
spy_bm['r_market'] = spy_bm['PX_LAST'].pct_change()

# Rate shift — simulated parallel USD curve move
np.random.seed(42)
rate_series = pd.Series(
    np.random.normal(0, 0.0005, len(spy_bm)),
    index=spy_bm.index, name='dy'
)

# FX
usd = BBG.bdh('EURUSD Curncy', 'PX_LAST', START_DATE, VALUATION_DATE_STR)
usd['r_fx_USD'] = usd['PX_LAST'].pct_change()

# Market moves
market_moves = pd.DataFrame(index=spy_bm.index)
market_moves['r_market'] = spy_bm['r_market']
market_moves['dy']       = rate_series
market_moves['r_fx_USD'] = usd['r_fx_USD']
market_moves = market_moves.dropna()

# Daily positions history
with ENGINE.connect() as conn:
    positions_history = pd.read_sql(sa.text("""
        SELECT p.date, p.isin, p.asset_class, p.currency,
               p.market_value_eur, pe.beta, pe.dur_adj_mid
        FROM positions p
        LEFT JOIN positions_enriched pe
            ON p.isin = pe.isin AND p.fund_id = pe.fund_id
        WHERE p.fund_id = :fund_id
        ORDER BY p.date
    """), conn, params={'fund_id': FUND_ID})

positions_history['date'] = pd.to_datetime(positions_history['date'])

common_dates     = market_moves.index.intersection(pnl_actual.index)
market_moves_aln = market_moves.loc[common_dates]
returns_aligned      = pnl_actual.loc[common_dates]
pos_history_aln  = positions_history[positions_history['date'].isin(common_dates)]

attr = compute_pnl_attribution(pos_history_aln, market_moves_aln, returns_aligned)

# Cumulative attribution chart
fig, ax = plt.subplots(figsize=(8, 4))
cum = attr[['pnl_equity', 'pnl_rates', 'pnl_fx', 'pnl_residual']].cumsum() / 1e6
ax.plot(cum.index, cum['pnl_equity'],   color=ACCENT,    linewidth=1.5, label='Equity')
ax.plot(cum.index, cum['pnl_rates'],    color=ACCENT2,   linewidth=1.5, label='Rates')
ax.plot(cum.index, cum['pnl_fx'],       color=ACCENT3,   linewidth=1.5, label='FX')
ax.plot(cum.index, cum['pnl_residual'], color=C['red'], linewidth=1.0,
        linestyle='--', label='Residual')
ax.axhline(0, color='white', linewidth=0.5, linestyle='--')
ax.set_ylabel('Cumulative P&L (EUR MM)')
section_title(ax, f'Cumulative P&L Attribution by Risk Factor — UCITS', fontsize=16)
ax.legend(fontsize=9)
plt.tight_layout()
save_fig(fig, FUND_ID, "09. PnL attribution UCITS")
plt.show()


# Model quality
RESIDUAL_THRESHOLD_PCT = 0.20
resid_vol = attr['pnl_residual'].std()
flagged = attr[
    (attr['pct_explained'].notna()) &
    (
        (1 - attr['pct_explained'] > RESIDUAL_THRESHOLD_PCT) |
        (attr['pnl_residual'].abs() > 2.0 * resid_vol)
    )
].copy()

print(f"{'Attribution period':<35} {attr.index.min().date()} to {attr.index.max().date()}")
print(f"{'Days attributed':<35} {len(attr)}")
print(f"{'Correlation (actual vs expl.)':<35} {attr['pnl_actual'].corr(attr['pnl_explained']):.3f}")
print(f"{'Median % explained':<35} {attr['pct_explained'].median():.1%}")
print(f"{'Days >= 80% explained':<35} {(attr['pct_explained'] >= 0.80).sum()} ({(attr['pct_explained'] >= 0.80).mean():.1%})")
print(f"{'Residual vol (EUR)':<35} {attr['pnl_residual'].std():,.0f}")
print(f"{'Residual / total vol':<35} {attr['pnl_residual'].std() / attr['pnl_actual'].std():.1%}")
print(f"{'Flagged days':<35} {len(flagged)} ({len(flagged)/len(attr):.1%})")
print()
print("Note: residual = idiosyncratic return not explained by market beta.")
print("For a long/short equity fund the residual represents alpha —")
print("stock-specific return from security selection independent of")
print("market direction. Persistent positive residual = successful stock picking.")




**Methodology and limitations**

Sensitivity-based attribution using actual daily positions and market moves.
Regression-based attribution is not used — it gives average historical loadings
and cannot reflect current position composition.

Benchmark: SX5E (Euro Stoxx 50) — appropriate for this EUR-denominated balanced fund.

**Limitations:**
* Single-factor equity model — no sector, style, or stock-level decomposition
* Rate attribution uses a simulated parallel shift — no key-rate DV01
* FX covers EUR/USD only
* Position composition is static in this simulation

**Regulatory context:** this section is an internal governance output consumed
by the Board and risk committee. It supports the AIFMD Article 15 evidence
pack and CSSF expectations for risk management reporting. It is not a direct
Annex IV or Annex VI deliverable.

<!-- Blank markdown cell retained to preserve notebook structure during markdown-only cleanup. -->